Instrucciones:

* Entrenar una CNN simple con batch sizes de: 16, 64, 128, 512.
* Registrar precisión y tiempo de entrenamiento.
* Comparar la estabilidad y velocidad de convergencia.

Solución en Keras

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import time

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train[..., None] / 255.0
x_test = x_test[..., None] / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
batch_sizes = [16, 64, 128, 512]

In [4]:
for batch in batch_sizes:
    print(f"\n🔍 Batch size: {batch}")

    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    start = time.time()
    model.fit(x_train, y_train, epochs=3, batch_size=batch, verbose=0)
    duration = time.time() - start

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"⏱ Tiempo: {duration:.2f}s | 🎯 Accuracy: {acc:.4f}")


🔍 Batch size: 16


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


⏱ Tiempo: 117.67s | 🎯 Accuracy: 0.9840

🔍 Batch size: 64
⏱ Tiempo: 66.89s | 🎯 Accuracy: 0.9816

🔍 Batch size: 128
⏱ Tiempo: 61.47s | 🎯 Accuracy: 0.9810

🔍 Batch size: 512
⏱ Tiempo: 124.54s | 🎯 Accuracy: 0.9787


Solución en Pytorch

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

In [6]:
transform = transforms.ToTensor()
trainset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
testset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.03MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 158kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.07MB/s]


In [7]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 32, 3)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(32*13*13, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv(x)))
        x = x.view(-1, 32*13*13)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [8]:
batch_sizes = [16, 64, 128, 512]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
for batch in batch_sizes:
    print(f"\n🔍 Batch size: {batch}")
    trainloader = DataLoader(trainset, batch_size=batch, shuffle=True)
    testloader = DataLoader(testset, batch_size=1000)

    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())

    start = time.time()
    model.train()
    for epoch in range(3):
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    duration = time.time() - start

    # Evaluar
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f"⏱ Tiempo: {duration:.2f}s | 🎯 Accuracy: {accuracy:.4f}")


🔍 Batch size: 16
⏱ Tiempo: 164.70s | 🎯 Accuracy: 0.9839

🔍 Batch size: 64
⏱ Tiempo: 103.30s | 🎯 Accuracy: 0.9818

🔍 Batch size: 128
⏱ Tiempo: 95.02s | 🎯 Accuracy: 0.9788

🔍 Batch size: 512
⏱ Tiempo: 130.12s | 🎯 Accuracy: 0.9720


Actividades:

* Graficar precisión vs. batch size.
* ¿Cuál fue el batch size con mejor precisión? ¿Y con menor tiempo?
* ¿Qué ventajas tiene un batch grande o pequeño en memoria y convergencia?